# CRNS N0 Calibration

**Yuxi Zhang** · Oct 25, 2026 · [@yuxi-research](https://github.com/yuxi-research)

📘 This notebook calibrates the parameter (N0) in the CRNS soil moisture conversion function.

⚙️ **Main Processing Steps (Calibration)**  
1. Load and pair rover and soil data
   * Read the corrected rover data kriged at sample locations
   * Read soil moisture and water-equivalents data from soil sample data.
   * Pair rover data and soil data.
   * Rows with missing data in soil moisture is removed.
   * Read kriged water-equivalents data.

2. Conversion functions
   * Define conversion functions
   * Calculate required variables in the conversion function.
   * If water-equivalents data is missing at some points, fill NaN using the kriged data.

3. Calibration
   * Define a calibration function
   * Calibrate N0 using all dates
   * Calibrate N0 for each dates (optimal)
   * Calibrate N0 using selected dates

4. Visualise results


💬 **Notes**

📥 **Input**  

- `./data_lv2/kriging`
  - `kriged_ok/`  
  - `kriged_b2pk/`
  - `kriged_p2pk/`  

📤 **Output**  
- `./data_lv2<farm_name>/calibration`
  - `caliN0_b2pk/`  
  - `caliN0_p2pk/`  


### Settings

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import re
import datetime as dt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from utils.calc_metrics import calc_lccc, calc_corr, calc_metrics
from utils.rover_process import extract_raster_values
save_pdf = True
display=True


In [ ]:
# Path settings
from global_settings import *
farm_name = os.environ.get("FARM_NAME", "Amaroo")
dirs = ProjectPaths(farm_name=farm_name)
proj_crs = dirs.proj_crs
inset_loc = dirs.inset_loc
buffer_dis = BUFFER_DIS


#### Pipeline parameters

In [ ]:
# Parameters
try:
    mode
except NameError:
    mode = 'b2pk'

try:
    remove_anomalies
except NameError:
    remove_anomalies = True
    # If remove_anomalies is True, the code will filter out anomalies (A03 2022 and D08) from the data before plotting and calculating metrics. Output is saved in a seperate folder with '_clean' suffix.
    # If False, it will use the original data without filtering.
    # This doesn't affect the calibration process as the filter is applied after the calibration is done.
    # The anomalies in calibration process is controlled by removing outliers.

## Remove A03 and D08
# gdf_paired = gdf_paired[~((gdf_paired['SiteID'] == 'A03' ) & (gdf_paired['CalibrationDate'].apply(lambda x: x.year) == 2022))]
# gdf_paired = gdf_paired[~(gdf_paired['SiteID'] == 'D08')]

In [ ]:
dir_ok = dirs.lv2_ok

dir_krig = dirs.lv2_krig / f'kriged_{mode}'
dir_out = dirs.lv2_cal / f'cal_{mode}'

if remove_anomalies:
    dir_out = dir_out.with_name(dir_out.name + '_clean')

dir_out_f = dir_out / "figures"
dir_out_f.mkdir(parents=True, exist_ok=True)

In [ ]:
dir_krig

## Read all data
#### Search the directory to find the kriged rover data

In [ ]:
# Krieged counts values at sampling locations
filenames_counts = [f for f in os.listdir(dir_krig) if re.match(rf'^{farm_name}_.*_CountsCorr_points_{mode}.csv$', f)]
filenames_counts.sort()
filenames_counts

In [ ]:
df_rover = []
for filename in filenames_counts:
    farm_name, date_str, _, _, _ = re.split( '_', filename)
    date = dt.datetime.strptime(date_str, '%Y-%m-%d').date()
    # Read rover data with corrected counts
    df = pd.read_csv(os.path.join(dir_krig, filename)).astype({'CountsCorr_kriged': float, 'kriging_variance': float})
    df['Date'] = date
    print(farm_name, date_str)
    df_rover.append(df)

df_rover = pd.concat(df_rover, ignore_index=True)

In [ ]:
df_rover

### Quality control 
* TODO: This will be removed in a later version, as the quality control should be performed in the B2PK stage

In [ ]:
# Remove outliers based on block measurement min and max, and kriging variance.
# Remove this in future version.
variance_threshold = 50
df_rover['flag_outlier'] = df_rover.apply(lambda x: (x['kriging_variance'] > x['var_p95']) or (x['CountsCorr_kriged'] < x['block_min']) or (x['CountsCorr_kriged'] > x['block_max']), axis=1)
print(f"Number of outliers: {df_rover['flag_outlier'].sum()}")

df_rover_masked = df_rover[~df_rover['flag_outlier']].copy()
print(f"Number of data points after removing outliers: {len(df_rover_masked)}")

In [ ]:
df_rover[df_rover['flag_outlier'] ]

In [ ]:
df_rover_masked = df_rover_masked[['SiteID', 'Date', 'CountsCorr_kriged', 'kriging_variance']]

In [ ]:
# Filter dates oustide the range of block measurements
df_rover_masked = df_rover_masked[df_rover_masked['Date'] < dt.date(2022, 7, 1)]

### Read files

#### Boundary

In [ ]:
# Read boundary
bound = gpd.read_file(os.path.join(dirs.lv1_shp, f'{farm_name}_boundary_epsg{proj_crs}.shp'))
bound = bound.to_crs(proj_crs)

#### Read all soil moisture data

In [ ]:
# Read soil moisture data with covariates
df_soil = pd.read_csv(os.path.join(dirs.lv2_data, f'{farm_name}_soilsamples_weighted.csv'), parse_dates=['Date'])
df_soil['Date'] = df_soil['Date'].dt.date
df_soil['geometry'] = gpd.points_from_xy(df_soil['Longitude'], df_soil['Latitude'], crs= '4326').to_crs(proj_crs)
gdf_soil = gpd.GeoDataFrame(df_soil, geometry='geometry')

print("Dropping rows with missing SMGrav or geometry")
gdf_soil = gdf_soil.dropna(subset=['SMGrav', 'geometry'])

#### Pairing rover counts with soil sampling data for calibraiton

In [ ]:
# Merge by SiteID
gdf_paired = gdf_soil.merge(df_rover_masked, on=['SiteID', 'Date'], suffixes=(None, '_rover'))
gdf_paired = gdf_paired.sort_values('Date')
# Spatial join to find the nearest point. Don't need this anymore since we have the same points in rover and soil data, but keep it for checking distance.
# gdf_paired = gdf_soil_date.sjoin_nearest(df_rover, how='left', distance_col='distance', rsuffix='rover')

In [ ]:
df_rover_masked

#### Extract water-equivalent values from ordinary kriged maps


In [ ]:
import rioxarray as rxr
import rasterio

# Extract soil moisture and water-equivalent values from ordinary kriged raster files at the sampling locations.
for col in ['watLatticeGrav', 'watHydrogenGrav', 'watCarbonGrav', 'watLatticeVol', 'watHydrogenVol', 'watCarbonVol']:
    try:
        rxrdata = rxr.open_rasterio(os.path.join(dir_ok, f'{farm_name}_{date_str}_{col}_weighted_ok.tif'),
                                    mask_and_scale=True).squeeze()
        
        # Create a separate column for the extracted raster values to avoid overwriting the original column.
        gdf_paired[f'{col}_ok'] = gdf_paired[['geometry']].apply(extract_raster_values, raster=rxrdata)

    except rasterio.errors.RasterioIOError: 
        print(f'File for {col} not found, skipping.')

## Conversion function

The equation below is used to characterise the relationship between neutron intensity and soil moisture
$$ \theta = [\frac{0.0808} {\frac{N}{N_0} -0.372} - 0.115 - w_{lat} - w_{soc}] \cdot \rho_{bd} $$

where 𝜃 is the volumetric water content (cm3 cm-3), 𝑁 is the corrected neutron intensity (counts s-1),
N0 is the neutron intensity above the dry soil, 𝑤𝑙𝑎𝑡 is lattice water content, 𝑤𝑠𝑜𝑐 is soil
organic matter, and 𝜌𝑏𝑑 is the bulk density of the soil. N0 parameter is determined by field calibration.

The lattice water is the amount of water released at 1000 °C. Since this is an expensive practice, we
adopted an alternative method from McJannet et al. (2017) who used a relationship between clay
content and lattice water, which was created by Greacen (1981).
$$ w_{lat} = 0.00075 \cdot clay + 0.0121 $$

where 𝑤𝑙𝑎𝑡 is the lattice water content (cm3 cm-3) and clay is the clay content (%). The relationship
We used the same linear relationship to predict our lattice water content across the farm with a soil
map of clay from Orton et al. (2016).

In [ ]:
# Define conversion functions to compute N0 and theta

def compute_N0(theta, CountsCorr, watEquivalent, bd=1):
    tmp = (0.0808 / (theta / bd + watEquivalent + 0.115) + 0.372)
    N0 = CountsCorr / tmp
    return N0

def compute_theta(N0, CountsCorr, watEquivalent, bd):
    tmp = (CountsCorr / N0) - 0.372
    tmp = ((0.0808 / tmp) - 0.115 - watEquivalent) * bd
    return tmp


In [ ]:
### Test the conversion functions with sample data
x0 = np.arange(200, 490, 10)
y0 = compute_theta(N0=500, CountsCorr=x0, watEquivalent=0.1, bd=1)
plt.figure(figsize=(4, 3))
plt.plot(x0, y0, marker='o')

### Calculate all required variables in the conversion function

In [ ]:
gdf_paired['CountsCorr'] = gdf_paired['CountsCorr_kriged']
print(f"Number of paired points: {len(gdf_paired)}")

In [ ]:
# If bulk density is not available, calculate gravimetric soil moisture. Otherwise, calculate volumetric soil moisture.
if ('BD' not in gdf_paired.columns) or (gdf_paired['BD'].mean() == 1):
    gdf_paired['BD'] = 1
    var_type = 'Grav'
else:
    var_type = 'Vol'

In [ ]:
# Check for any missing values in the relevant columns
gdf_paired[[c for c in gdf_paired.columns if var_type in c]].isnull().sum()

In [ ]:
# Fill missing values in watLattice, watHydrogen, and watCarbon columns with their corresponding '_ok' columns if they exist.
# This ensure the sampled values are used first, and the kriged values are used as a backup for missing values.
# SM are never filled with kriged values, only the water-equivalent values are filled with kriged values.

for col in [f'watHydrogen{var_type}', f'watCarbon{var_type}', f'watLattice{var_type}']:
    if (f'{col}_ok' in gdf_paired.columns) and (col in gdf_paired.columns):
        gdf_paired[col].fillna(gdf_paired[f'{col}_ok'], inplace=True)

# Drop the '_ok' columns after filling missing values
gdf_paired.drop(columns=[c for c in gdf_paired.columns if c.endswith('_ok')], inplace=True)


In [ ]:
# Calculate equivalent water content from lattice and carbon content. If hydrogen content is available, use it instead.
if (f'watHydrogen{var_type}' in gdf_paired.columns) and (gdf_paired[f'watHydrogen{var_type}'].notna().sum() > 0):
    gdf_paired[f'watEquivalent{var_type}'] = gdf_paired[f'watHydrogen{var_type}'].copy()
    if (f'watLattice{var_type}' in gdf_paired.columns) and (f'watCarbon{var_type}' in gdf_paired.columns):
        # Fill missing values in watEquivalent with the sum of watLattice and watCarbon
        gdf_paired[f'watEquivalent{var_type}'].fillna(gdf_paired[f'watLattice{var_type}'] + gdf_paired[f'watCarbon{var_type}'], inplace=True)
else:
    gdf_paired[f'watEquivalent{var_type}'] = gdf_paired[f'watLattice{var_type}'] + gdf_paired[f'watCarbon{var_type}']



In [ ]:
# Check for any remaining missing values in the relevant columns
gdf_paired[[c for c in gdf_paired.columns if var_type in c]].isnull().sum()

## Calibration
### Defined a calibration function

In [ ]:
# Define calibration function
def calibrate_N0(gdf_cal, vartype='Grav', date_str="", outlier=0.05):

    ###############################################
    # Remove outliers
    qmin = gdf_cal['CountsCorr'].quantile(outlier/2)
    qmax = gdf_cal['CountsCorr'].quantile(1-outlier/2)
    gdf_cal['CountsCorr'] = gdf_cal['CountsCorr'].map(lambda x: x if (x >= qmin and x <= qmax) else np.nan)
    ncounts = len(gdf_cal)
    gdf_cal = gdf_cal.dropna(subset=['CountsCorr'])
    print(f"Removed {ncounts - len(gdf_cal)} outliers based on CountsCorr, remaining {len(gdf_cal)} points for calibration.")
    if len(gdf_cal) <= 2:
        print("Not enough points for calibration after removing outliers, skipping.")
        return gdf_cal, (None, None, None, None)
    ###############################################
    # Back-calculate N0 using point mean
    n0_point_mean = compute_N0(gdf_cal[f'SM{vartype}'].mean(), 
                               gdf_cal['CountsCorr'].mean(), 
                               gdf_cal[f'watEquivalent{vartype}'].mean(), 
                               gdf_cal['BD'].mean())

    gdf_cal['theta_point_mean'] = compute_theta(n0_point_mean, gdf_cal['CountsCorr'], gdf_cal[f'watEquivalent{vartype}'], gdf_cal['BD'])
    rmse_point_mean = ((gdf_cal[f'SM{vartype}'] - gdf_cal['theta_point_mean'])**2).mean()**0.5

    ###############################################
    # Back-calculate N0 using all points
    gdf_cal['N0'] = compute_N0(gdf_cal[f'SM{vartype}'], gdf_cal['CountsCorr'], gdf_cal[f'watEquivalent{vartype}'], gdf_cal['BD'])
    n0_all_points = gdf_cal['N0'].mean()
    gdf_cal['theta_all_points'] = compute_theta(n0_all_points, gdf_cal['CountsCorr'], gdf_cal[f'watEquivalent{vartype}'], gdf_cal['BD'])
    rmse_all_points = ((gdf_cal[f'SM{vartype}'] - gdf_cal['theta_all_points'])**2).mean()**0.5

    ###############################################

    # Optimise N0 using all rover points
    # Search N0: two rounds of search (coarse + fine)
    best_rmse = float('inf')
    best_n0 = None

    results1 = []
    # First round of coarse search: N0 1 ~ 1000
    for N0 in np.arange(1, 1001, 1):
        gdf_cal['theta'] = compute_theta(N0, gdf_cal['CountsCorr'], gdf_cal[f'watEquivalent{vartype}'], gdf_cal['BD']).copy()
        rmse = ((gdf_cal[f'SM{vartype}'] - gdf_cal['theta'])**2).mean()**0.5
        lccc = calc_lccc(gdf_cal[f'SM{vartype}'], gdf_cal['theta'])
        corr = calc_corr(gdf_cal['theta'], gdf_cal[f'SM{vartype}'])
        results1.append((N0, rmse, lccc, corr))
        if rmse < best_rmse:
            best_rmse = rmse
            best_n0 = N0

    results2 = []
    # Second round of fine search: around best_n0 ± 1
    for N0 in np.arange(best_n0 - 1, best_n0 + 1, 0.001):
        gdf_cal['theta'] = compute_theta(N0, gdf_cal['CountsCorr'], gdf_cal[f'watEquivalent{vartype}'], gdf_cal['BD']).copy()
        rmse = ((gdf_cal[f'SM{vartype}'] - gdf_cal['theta'])**2).mean()**0.5
        lccc = calc_lccc(gdf_cal[f'SM{vartype}'], gdf_cal['theta'])
        corr = calc_corr(gdf_cal['theta'], gdf_cal[f'SM{vartype}'])
        results1.append((N0, rmse, lccc, corr))
        if rmse < best_rmse:
            best_rmse = rmse
            best_n0 = N0

    print(f"Best N0 = {best_n0:.3f}, with RMSE = {rmse:.4f}, LCCC = {lccc:.4f}, corr = {corr:.3f}")
    gdf_cal[f'SM{vartype}_pred'] = compute_theta(best_n0, gdf_cal['CountsCorr'], gdf_cal[f'watEquivalent{vartype}'], gdf_cal['BD'])

    # Check the metric when N0 varies
    results1 = pd.DataFrame(results1, columns=['N0', 'RMSE', 'LCCC', 'corr'])
    results1.sort_values('N0', inplace=True)
    # sns plot two y curves in one dataframe
    plt.figure(figsize=(5.3, 4))
    ax = plt.gca()
    ax.plot(results1['N0'], results1['RMSE'], label='RMSE', color='black', lw=2)
    
    ax.plot(results1['N0'], results1['corr'], label='Correlation', color='deeppink', ls = '--', lw=2)
    ax.plot(results1['N0'], results1['LCCC'], label='LCCC', color='green', ls = '-.', lw=2)

    # ax.hlines(0, xmin=results1['N0'].min(), xmax=results1['N0'].max(), color='grey', lw=0.5, ls='--')
    ax.set_xlabel(r'$\mathrm{N}_0\ (\mathrm{cpm})$ ')
    ax.set_ylabel('Metric values')
    ax.set_ylim(0, 0.55)
    ax.legend(loc='upper left')
    ax.get_legend().remove() # If remove legend
    ax.set_title(f'{farm_name} {date_str} ({mode.upper()})')
    plt.tight_layout()

    plt.savefig(os.path.join(dir_out, f'{farm_name}_{date_str}_{mode}_N0_calibration.png'), bbox_inches='tight')
    if save_pdf:
        plt.savefig(os.path.join(dir_out_f, f'{farm_name}_{date_str}_{mode}_N0_calibration.pdf'), bbox_inches='tight')
    
    plt.show() if display else plt.close()

    return gdf_cal, (best_n0, n0_point_mean, n0_all_points, best_rmse, rmse_point_mean, rmse_all_points)
    # n0_optimized = best_n0
    # gdf_cal_by_date.append(gdf_cal)
    # n0_by_date.append([date, n0_optimized, n0_point_mean, n0_all_points, best_rmse])

### Calibrate N0 using all dates

In [ ]:
dates = gdf_paired['Date'].unique()
dates.sort()
print('All dates:', dates)

In [ ]:
# Calibrate N0 using all dates
n0_all = []
gdf_cal_all_dates, results = calibrate_N0(gdf_paired.copy(), vartype=var_type, date_str="All Dates", outlier=0.1)
n0_all.append(['All Dates', *results, dates])


### Calibrate N0 for each dates

In [ ]:
# For each of the dates, calibrate N0 and compute theta.
n0_by_date = []
gdf_cal_by_date = []
for date in dates:
    date_str = date.strftime('%Y-%m-%d')
    gdf_cal = gdf_paired[gdf_paired['Date'] == date].copy()
    if len(gdf_cal) == 0:
        print(f"No data for date {date}, skipping calibration.")
        continue
    
    gdf_cal, results = calibrate_N0(gdf_cal, vartype=var_type, date_str=date_str, outlier=0.1)
    gdf_cal_by_date.append(gdf_cal)
    n0_by_date.append(["By Date", *results, date])

n0_all.extend(n0_by_date)

### Calibrate N0 using selected dates

In [ ]:
dates_selected = [date for date in dates if (date < dt.date(2022, 7, 1))]
print('Selected dates:', dates_selected)

In [ ]:
# Calibrate N0 using selected dates
gdf_cal_selected_dates = gdf_paired[gdf_paired['Date'].isin(dates_selected)].copy()
gdf_cal_selected_dates, results = calibrate_N0(gdf_cal_selected_dates, vartype=var_type, date_str="Selected Dates", outlier=0.1)
n0_all.append(['Selected Dates', *results, dates_selected])

In [ ]:
n0_by_date = pd.DataFrame(n0_by_date, columns=['CalibrationType', 'N0_optimized', 'N0_point_mean', 'N0_all_points', 'Best_RMSE', 'RMSE_point_mean', 'RMSE_all_points', 'CalibrationDate'])
n0_all = pd.DataFrame(n0_all, columns=['CalibrationType', 'N0_optimized', 'N0_point_mean', 'N0_all_points', 'Best_RMSE', 'RMSE_point_mean', 'RMSE_all_points', 'CalibrationDate'])

n0_by_date['Farm'] = farm_name
n0_all['Farm'] = farm_name

# n0_by_date.to_csv(os.path.join(pth_out_cal, f'{farm_name}_N0_calibrated_by_date.csv'), index=False)
n0_all.to_csv(dir_out / f'{farm_name}_N0_calibrated_{mode}.csv', index=False)

### Check calibration result using calibrated N0 of each survey


In [ ]:
gdf_paired = gdf_paired.merge(n0_by_date, left_on='Date', right_on='CalibrationDate', how='left')

In [ ]:
gdf_paired[f'SM{var_type}_pred'] = compute_theta(gdf_paired['N0_optimized'], gdf_paired['CountsCorr'], gdf_paired[f'watEquivalent{var_type}'], gdf_paired['BD'])
gdf_paired.to_csv(os.path.join(dir_out, f'{farm_name}_caliN0_{mode}.csv'), index=False)


In [ ]:
if remove_anomalies:
    # Remove A03 2022 and D08
    gdf_paired = gdf_paired[~((gdf_paired['SiteID'] == 'A03' ) & (gdf_paired['CalibrationDate'].apply(lambda x: x.year) == 2022))]
    gdf_paired = gdf_paired[~(gdf_paired['SiteID'] == 'D08')]

In [ ]:

metrics = gdf_paired.groupby('Date').apply(lambda x: calc_metrics(x[f'SM{var_type}'], x[f'SM{var_type}_pred'])).apply(pd.Series)
if len(metrics) > 1:
    metrics_overall = gdf_paired.groupby('Farm').apply(lambda x: calc_metrics(x[f'SM{var_type}'], x[f'SM{var_type}_pred'])).apply(pd.Series)
    metrics_overall['Date'] = 'Overall'
    metrics_overall = metrics_overall.reset_index().set_index('Date').drop('Farm', axis=1)
    metrics = pd.concat([metrics, metrics_overall], ignore_index=False, axis=0)


metrics['Farm'] = farm_name
metrics.to_csv(dir_out / f'{farm_name}_evaluation_metrics_{mode}.csv')


### Check calibration result using all dates


In [ ]:
n0_all

In [ ]:
selected_n0 = n0_all[n0_all['CalibrationType'] == 'Selected Dates']['N0_optimized'].iloc[0]

In [ ]:
gdf_paired[f'SM{var_type}_pred2'] = compute_theta(selected_n0, gdf_paired['CountsCorr'], gdf_paired[f'watEquivalent{var_type}'], gdf_paired['BD'])

metrics2 = gdf_paired.groupby('Date').apply(lambda x: calc_metrics(x[f'SM{var_type}'], x[f'SM{var_type}_pred2'])).apply(pd.Series)
if len(metrics2) > 1:
    metrics2_overall = gdf_paired.groupby('Farm').apply(lambda x: calc_metrics(x[f'SM{var_type}'], x[f'SM{var_type}_pred2'])).apply(pd.Series)
    metrics2_overall['Date'] = 'Overall'
    metrics2_overall = metrics2_overall.reset_index().set_index('Date').drop('Farm', axis=1)
    metrics2 = pd.concat([metrics2, metrics2_overall], ignore_index=False, axis=0)
    metrics2

metrics2['Farm'] = farm_name
metrics2.to_csv(dir_out / f'{farm_name}_evaluation_metrics2_{mode}.csv')


### Plot


In [ ]:
vmin =  gdf_paired[[f'SM{var_type}', f'SM{var_type}_pred', f'SM{var_type}_pred2']].min().min() - 0.05
vmax = gdf_paired[[f'SM{var_type}', f'SM{var_type}_pred', f'SM{var_type}_pred2']].max().max() + 0.05
unit = r'$\mathrm{m}^3\ \mathrm{m}^{-3}$' if var_type == 'Vol' else r'$\mathrm{g}\ \mathrm{g}^{-1}$'


In [ ]:
plt.figure(figsize=(5.3, 5.3))
sns.scatterplot(x=gdf_paired[f'SM{var_type}'], y=gdf_paired[f'SM{var_type}_pred'], hue=gdf_paired['Date'], style=gdf_paired['Date'], 
                markers=['o', 's', '^'], s=100, alpha=0.9, palette='Set1')
ax = plt.gca()

# Annotate points with SiteID
for i, row in gdf_paired.iterrows():
    ax.annotate(row['SiteID'], (row[f'SM{var_type}'], row[f'SM{var_type}_pred']),
                xytext=(2, 2), fontsize=12, textcoords='offset points', color='#333333', alpha=0.9, zorder=10)

# Statistics text
fig_text = []
for i, row in metrics.iterrows():
    text = f"{row.name}:\nRMSE: {row['RMSE']:.3f}, LCCC: {row['LCCC']:.2f}, r: {row['corr']:.2f}"
    fig_text.append(text)

fig_text = '\n'.join(fig_text)

# ax.text(0.05, 0.95, fig_text,
#         transform=ax.transAxes, color='black', verticalalignment='top', horizontalalignment='left',
#         fontsize=16, bbox=dict(facecolor='white', alpha=0.6, edgecolor='black', boxstyle='square,pad=0.5'))

### Axes settings
ax.set_xlabel(f'Sampled SM {var_type} ({unit})')
ax.set_ylabel(f'Derived SM {var_type} ({unit})')

ax.set_title(f'{farm_name} ({mode.upper()})')
ax.plot([-1, 1], [-1, 1], color='grey', linestyle='--', linewidth=1)
# ax.grid(True)
ax.set_xlim(vmin, vmax)
ax.set_ylim(vmin, vmax)
ax.set_aspect('equal', adjustable='box')
ax.legend(loc='lower right')
outfile = f'{farm_name}_caliN0_singledate_{mode}.png'
plt.savefig(dir_out / outfile, bbox_inches='tight')
if save_pdf:
    plt.savefig(dir_out_f / outfile.replace('.png', '.pdf'), bbox_inches='tight')
plt.show()

for i, row in metrics.iterrows():
    text = f"{row.name}:\nRMSE: {row['RMSE']:.3f}, LCCC: {row['LCCC']:.2f}, corr: {row['corr']:.2f}, bias: {row['bias']:.2f}, ubRMSE: {row['ubRMSE']:.2f}, mean: {row['mean']:.2f}, N: {row['Sample size']}"
    print(text)

In [ ]:
plt.figure(figsize=(5.3, 5.3))
sns.scatterplot(x=gdf_paired[f'SM{var_type}'], y=gdf_paired[f'SM{var_type}_pred2'], hue=gdf_paired['Date'], style=gdf_paired['Date'], 
                markers=['o', 's', '^'], s=100, alpha=0.9, palette='Set1')
ax = plt.gca()

# Annotate points with SiteID
for i, row in gdf_paired.iterrows():
    ax.annotate(row['SiteID'], (row[f'SM{var_type}'], row[f'SM{var_type}_pred2']),
                xytext=(2, 2), fontsize=12, textcoords='offset points', color='#333333', alpha=0.9, zorder=10)

# Statistics text
fig_text = []
for i, row in metrics2.iterrows():
    text = f"{row.name}:\nRMSE: {row['RMSE']:.3f}, LCCC: {row['LCCC']:.2f}, r: {row['corr']:.2f}"
    fig_text.append(text)

fig_text = '\n'.join(fig_text)

# ax.text(0.05, 0.95, fig_text,
#         transform=ax.transAxes, color='black', verticalalignment='top', horizontalalignment='left',
#         fontsize=16, bbox=dict(facecolor='white', alpha=0.6, edgecolor='black', boxstyle='square,pad=0.5'))

### Axes settings
ax.set_xlabel(f'Sampled SM {var_type} ({unit})')
ax.set_ylabel(f'Derived SM {var_type} ({unit})')

ax.set_title(f'{farm_name} ({mode.upper()})')
ax.plot([-1, 1], [-1, 1], color='grey', linestyle='--', linewidth=1)
# ax.grid(True)
ax.set_xlim(vmin, vmax)
ax.set_ylim(vmin, vmax)
ax.set_aspect('equal', adjustable='box')
ax.legend(loc='lower right')
outfile = f'{farm_name}_caliN0_alldates_{mode}.png'
plt.savefig(dir_out / outfile, bbox_inches='tight')
if save_pdf:
    plt.savefig(dir_out_f / outfile.replace('.png', '.pdf'), bbox_inches='tight')
plt.show()


for i, row in metrics2.iterrows():
    text = f"{row.name}:\nRMSE: {row['RMSE']:.3f}, LCCC: {row['LCCC']:.2f}, corr: {row['corr']:.2f}, bias: {row['bias']:.2f}, ubRMSE: {row['ubRMSE']:.2f}, mean: {row['mean']:.2f}, N: {row['Sample size']}"
    print(text)